## POC First trial run

### Cleaning and normalizing

In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df=pd.read_csv("datasets/syn-company_transactions_1000.csv")
df.head()

,transaction_id,date,description,amount,vendor,raw_category
0,TXN0001,09-12-2025,Awfis payment for rent business,42976.56,Awfis,Rent
1,TXN0002,09-12-2025,Zomato payment for meals wait,33540.48,Zomato,Meals
2,TXN0003,09-12-2025,Adobe payment for software because,4670.02,Adobe,Software
3,TXN0004,09-12-2025,Airtel payment for utilities beautiful,23616.59,Airtel,Utilities
4,TXN0005,09-12-2025,KSEB payment for utilities page,36346.58,KSEB,Utilities


In [3]:
df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y", errors="coerce")

In [4]:
df = df.dropna(subset=["date"])

In [5]:
def clean_text(s):
    if pd.isna(s):
        return ""
    s=s.lower()
    s=re.sub(r"[^a-z0-9\s]", " ", s)
    s=re.sub(r"\s+", " ", s).strip()
    return s
    

In [6]:
df["description_clean"]=df["description"].apply(clean_text)
df["vendor_clean"] = df["vendor"].str.lower().str.strip()

In [7]:
# Amount sanity: keep only positive amounts
df = df[df["amount"] > 0]

# Optional: clip extreme outliers (e.g., 99th percentile)
upper = df["amount"].quantile(0.99)
df["amount_clipped"] = df["amount"].clip(upper=upper)

### Feature Extraction

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

df["month"] = df["date"].dt.month
df["dow"] = df["date"].dt.dayofweek  # 0=Monday

X = df[["description_clean", "vendor_clean", "amount_clipped", "month", "dow"]]
y = df["raw_category"]

text_col = "description_clean"
cat_cols = ["vendor_clean"]
num_cols = ["amount_clipped", "month", "dow"]

preprocess = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(max_features=3000, ngram_range=(1,2)), text_col),
        ("cat", OneHotEncoder(handle_unknown="ignore", max_categories=20), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


### Transaction classification

Logistic Regression (fast, explainable).

XGBoost or RandomForest (non‑linear, handles interactions).

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

clf = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=1000, n_jobs=-1))
    ]
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))


                 precision    recall  f1-score   support

    IT Services       1.00      1.00      1.00        28
          Meals       1.00      1.00      1.00        34
Office Supplies       1.00      1.00      1.00        30
           Rent       1.00      1.00      1.00        23
       Software       1.00      1.00      1.00        26
         Travel       1.00      1.00      1.00        29
      Utilities       1.00      1.00      1.00        30

       accuracy                           1.00       200
      macro avg       1.00      1.00      1.00       200
   weighted avg       1.00      1.00      1.00       200



In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

rf_clf = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            n_jobs=-1,
            random_state=42
        ))
    ]
)

rf_clf.fit(X_train, y_train)
rf_pred = rf_clf.predict(X_test)

print("LogReg macro F1:", f1_score(y_test, y_pred, average="macro"))
print("RF macro F1:", f1_score(y_test, rf_pred, average="macro"))


LogReg macro F1: 1.0
RF macro F1: 1.0


### Tax‑engine‑friendly labeling

In [11]:
gst_rules = {
    "Meals":          {"gst_rate": 5,  "itc_eligible": "No"},
    "Travel":         {"gst_rate": 5,  "itc_eligible": "No"},  # for personal-type travel
    "Rent":           {"gst_rate": 18, "itc_eligible": "Yes"},
    "IT Services":    {"gst_rate": 18, "itc_eligible": "Yes"},
    "Software":       {"gst_rate": 18, "itc_eligible": "Yes"},
    "Office Supplies":{"gst_rate": 12, "itc_eligible": "Yes"},
    "Utilities":      {"gst_rate": 18, "itc_eligible": "Yes"},
    # extend as needed
}


In [12]:
def attach_tax_labels(df_input, model):
    X_all = df_input[["description_clean", "vendor_clean", "amount_clipped", "month", "dow"]]
    pred_cat = model.predict(X_all)
    df_out = df_input.copy()
    df_out["pred_category"] = pred_cat
    
    df_out["gst_rate"] = df_out["pred_category"].map(lambda c: gst_rules.get(c, {}).get("gst_rate", 0))
    df_out["itc_eligible"] = df_out["pred_category"].map(lambda c: gst_rules.get(c, {}).get("itc_eligible", "No"))
    return df_out

df_tax_ready = attach_tax_labels(df, clf)
df_tax_ready[["transaction_id", "date", "description", "amount", "pred_category", "gst_rate", "itc_eligible"]].head()


,transaction_id,date,description,amount,pred_category,gst_rate,itc_eligible
0,TXN0001,2025-12-09,Awfis payment for rent business,42976.56,Rent,18,Yes
1,TXN0002,2025-12-09,Zomato payment for meals wait,33540.48,Meals,5,No
2,TXN0003,2025-12-09,Adobe payment for software because,4670.02,Software,18,Yes
3,TXN0004,2025-12-09,Airtel payment for utilities beautiful,23616.59,Utilities,18,Yes
4,TXN0005,2025-12-09,KSEB payment for utilities page,36346.58,Utilities,18,Yes


### Forecasting with Prophet (basic feasibility POC)

In [13]:
from prophet import Prophet  # or `from prophet import Prophet` depending on install

# Aggregate to daily total amount
daily = df.groupby("date", as_index=False)["amount"].sum()
daily = daily.rename(columns={"date": "ds", "amount": "y"})

m = Prophet()  # you can add yearly/weekly seasonality later
m.fit(daily)

future = m.make_future_dataframe(periods=30)  # forecast next 30 days
forecast = m.predict(future)

forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail()


C:\Users\USER\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


ValueError: Dataframe has less than 2 non-NaN rows.

In [1]:
import pandas as pd

# Load the complete dataset
df = pd.read_parquet('datasets/down-0000.parquet')
print(f"Total records: {len(df):,}")
print(f"Columns: {list(df.columns)}")


Total records: 4,501,043
Columns: ['transaction_description', 'category', 'country', 'currency']


In [2]:
df.head()

,transaction_description,category,country,currency
0,Wage,Income,USA,USD
1,Arby's (Contactless),Food & Dining,AUSTRALIA,AUD
2,Occupational Therapy,Healthcare & Medical,USA,USD
3,Potbelly Store Branch,Food & Dining,UK,GBP
4,Amazon - AUSTRALIA,Shopping & Retail,AUSTRALIA,AUD


In [4]:
import pandas as pd

df1 = pd.read_csv("hf://datasets/HighkeyPrxneeth/BusinessTransactions/business_transactions_dataset.csv")

In [5]:
df1.head()

,Unnamed: 0,name,fsq_category_ids,Count,category_label,category_array,transaction_string,seed
0,0,Western Union,63be6904847c3692a84b9b3d,5993,Business and Professional Services > Financial...,"['Business and Professional Services', 'Financ...",WESTERN UNION #45678 SEND $29.99 TO CHICAGO IL...,1127428596
1,1,The PNC Financial Services Group,4bf58dd8d48988d10a951735,5032,Business and Professional Services > Financial...,"['Business and Professional Services', 'Financ...",The PNC Financial Services Group #7890 ACCOUNT...,292935982
2,2,Starbucks,4bf58dd8d48988d1e0931735,4893,"Dining and Drinking > Cafe, Coffee, and Tea Ho...","['Dining and Drinking', 'Cafe, Coffee, and Tea...",STARBUCKS COFFEE #98765 STORE 98765 PURCHASE $...,1047754538
3,3,Blue Rhino,63be6904847c3692a84b9b34,4818,Business and Professional Services > Chemicals...,"['Business and Professional Services', 'Chemic...","BLUE RHINO #4567 PURCHASE $1,250.00 FOR LABORA...",1537448773
4,4,Redbox,4bf58dd8d48988d126951735,4718,Retail > Video Store,"['Retail', 'Video Store']",Redbox #4567 DVD RENTAL $12.99 02/03/25 AZ,1248492378
